# Séance 5 — Décrire, comparer, conclure

**Analyse des données — L3 Économie**

Deux fichiers que vous connaissez :

- `cm03-salaires.csv`, le fichier du TD 3, pour décrire et comparer ;
- l'Enquête Emploi 2024 du TD 2, pour pondérer.

> *Exécution → Tout exécuter* avant de démarrer.

## 0. Les données, nettoyées

On reprend les décisions du TD 3 : montants convertis en nombres, code `999 999` mis en manquant, non-salariés écartés.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats

URL = ("https://raw.githubusercontent.com/StefaniaMarcassa/"
       "analyse_des_donnees/main/data/cm03-salaires.csv")

def en_nombre(serie):
    return pd.to_numeric(serie.astype(str).str.replace(" ", "", regex=False)
                                          .str.replace(",", ".", regex=False),
                         errors="coerce")

df = pd.read_csv(URL, sep=";", encoding="utf-8")
df["SALAIRE_NET"] = en_nombre(df["SALAIRE_NET"]).replace(999999, np.nan)
df = df[df["SALAIRE_NET"].isna() | (df["SALAIRE_NET"] > 0)].copy()

print(df.shape)

## 1. Décrire une distribution

### Trois nombres, trois questions

In [ ]:
df["SALAIRE_NET"].describe().round(0)

- La **moyenne** — 2 610 € — répond à : quel est le total, réparti également ?
- La **médiane** — 2 322 € — répond à : où se situe la personne du milieu ?
- Les **quantiles** répondent à : quelle est la forme de la distribution ?

La moyenne dépasse la médiane de près de 300 €. Ce n'est pas un défaut des données : c'est l'**asymétrie** de la distribution des salaires, tirée vers le haut par une minorité de hauts revenus. C'est un fait économique.

> Rapporter une moyenne sans la médiane, sur une variable asymétrique, est une faute.

### La règle des 1,5 écarts interquartiles

In [ ]:
q1, q3 = df["SALAIRE_NET"].quantile([0.25, 0.75])
iqr = q3 - q1
seuil_haut = q3 + 1.5 * iqr

print("seuil haut           :", round(seuil_haut), "euros")
print("points au-dessus     :", (df["SALAIRE_NET"] > seuil_haut).sum())
print("dont au plafond 10 000 :", (df["SALAIRE_NET"] == 10000).sum())

La règle signale **269** salaires au-dessus de 5 406 €. Sont-ils faux ? Non : ce sont des cadres, des dirigeants — des **valeurs extrêmes**, dans la logique de l'ensemble. Et 72 d'entre eux sont au plafond de diffusion, dont on ne connaît même pas la vraie valeur.

Le coefficient 1,5 est une **convention**, pas un résultat. Sur une distribution asymétrique, la règle signale mécaniquement des observations parfaitement valides.

> Elle sert à **regarder**, jamais à supprimer automatiquement.

## 2. Décrire par groupe

In [ ]:
df.groupby("SECTEUR").agg(
    salaire_moyen  = ("SALAIRE_NET", "mean"),
    salaire_median = ("SALAIRE_NET", "median"),
    effectif       = ("SALAIRE_NET", "count")
).round(0).sort_values("salaire_moyen", ascending=False)

Trois opérations en une : **découper** selon une clé, **appliquer** un calcul à chaque morceau, **recoller** les résultats.

L'unité statistique a changé : une ligne ne décrit plus un poste mais un secteur.

### Le piège de l'effectif

In [ ]:
r = df.groupby("AGE")["SALAIRE_NET"].agg(["mean", "count"])
r.sort_values("mean", ascending=False).head(3).round(0)

Les **68 ans** arrivent en tête, à 4 167 € — avec **sept** observations. Une moyenne sur sept personnes est extrêmement variable : le classement mesure surtout la taille des groupes.

> Un tableau agrégé s'accompagne **toujours** de l'effectif de chaque groupe.

## 3. Pondérer

Le fichier de salaires n'a pas de poids. L'Enquête Emploi, si : `EXTRIAN` dit combien de personnes chaque répondant représente. On ne charge que les deux colonnes utiles.

In [ ]:
URL_EEC = "https://www.insee.fr/fr/statistiques/fichier/8632441/FD_EEC_2024.parquet"
eec = pd.read_parquet(URL_EEC, columns=["ACTEU", "EXTRIAN"])

actifs  = eec[eec["ACTEU"].isin(["1", "2"])]
chomeur = (actifs["ACTEU"] == "2")

print("Taux de chomage, echantillon :", round(100 * chomeur.mean(), 2), "%")
print("Taux de chomage, population  :",
      round(100 * np.average(chomeur, weights=actifs["EXTRIAN"]), 2), "%")

**7,95 %** sans les poids, **7,44 %** avec — et c'est le second qui correspond au chiffre publié par l'INSEE.

Une moyenne calculée sans les poids décrit **l'échantillon**, pas la population. Elle est fausse pour toute question économique.

**Une conséquence désagréable.** La pondération complique aussi l'inférence : les erreurs-types calculées comme si l'échantillon était aléatoire simple sont sous-estimées. Un traitement rigoureux exige de connaître le plan de sondage. Nous pondérons les moyennes et les proportions, et nous **disons** que les erreurs-types sont approximatives.

## 4. Le tableau de comparaison

La question : les salaires des hommes et des femmes diffèrent-ils ?

> D'après le dictionnaire, `SEXE` vaut 1 pour les hommes et 2 pour les femmes — **vérifiez-le** : c'est le réflexe de toute la séance 2.

In [ ]:
df["SEXE"] = df["SEXE"].map({1: "Hommes", 2: "Femmes"})

df.groupby("SEXE")["SALAIRE_NET"].agg(["mean", "std", "count"]).round(0)

### Écart-type et erreur-type

Ce ne sont pas les mêmes objets.

- L'**écart-type** — 1 494 € chez les hommes — décrit la dispersion **des individus**.
- L'**erreur-type** décrit l'incertitude **sur la moyenne**. Elle diminue quand l'échantillon grandit.

In [ ]:
hommes = df.loc[df["SEXE"] == "Hommes", "SALAIRE_NET"].dropna()

ecart_type  = hommes.std()
erreur_type = ecart_type / np.sqrt(len(hommes))
moyenne     = hommes.mean()

print("ecart-type  :", round(ecart_type), "euros")
print("erreur-type :", round(erreur_type, 1), "euros")
print("IC a 95 %   :", (moyenne + np.array([-1.96, 1.96]) * erreur_type).round(0))

Un intervalle de confiance à 95 % dit : si l'on refaisait l'enquête un grand nombre de fois, 95 % des intervalles ainsi construits contiendraient la vraie moyenne.

Il ne dit **pas** que la vraie moyenne a 95 % de chances d'être dans cet intervalle-ci.

### Une fonction, pas trois copier-coller

In [ ]:
def resume(data, variable, groupe):
    g   = data.groupby(groupe)[variable]
    out = g.agg(["mean", "std", "count"])
    out["err_type"] = out["std"] / np.sqrt(out["count"])
    out["ic_bas"]   = out["mean"] - 1.96 * out["err_type"]
    out["ic_haut"]  = out["mean"] + 1.96 * out["err_type"]
    return out.round(1)

resume(df, "SALAIRE_NET", "SEXE")

C'est **le tableau que contient tout article empirique** : pour chaque groupe, la moyenne, la dispersion, l'effectif et l'incertitude.

Les deux intervalles — autour de 2 781 € et de 2 444 € — ne se chevauchent pas. Mais la bonne façon de comparer, c'est de tester l'**écart** lui-même.

## 5. Comparer deux groupes

In [ ]:
femmes = df.loc[df["SEXE"] == "Femmes", "SALAIRE_NET"].dropna()

test = stats.ttest_ind(hommes, femmes, equal_var=False)   # test de Welch

ecart = hommes.mean() - femmes.mean()
err   = np.sqrt(hommes.var() / len(hommes) + femmes.var() / len(femmes))

print("ecart        :", round(ecart), "euros")
print("IC de l'ecart:", (ecart + np.array([-1.96, 1.96]) * err).round(0))
print("p-value      :", f"{test.pvalue:.1e}")

Les hommes gagnent en moyenne **336 €** de plus par mois, avec un intervalle de confiance de **267 à 406 €**. La p-value est infime.

`equal_var=False` demande le **test de Welch**, qui ne suppose pas que les deux groupes ont la même dispersion. C'est le choix par défaut à retenir : il est juste quand les variances sont égales, et reste juste quand elles ne le sont pas.

**Ce que la p-value dit :** si les deux moyennes étaient égales dans la population, un écart aussi grand serait extrêmement improbable. **Ce qu'elle ne dit pas :** que l'écart est important, ni qu'il est dû au sexe — les hommes et les femmes du fichier n'ont ni les mêmes diplômes, ni les mêmes secteurs, ni le même temps de travail. Nous y reviendrons avec la régression.

### Significatif ne veut pas dire important

Une démonstration par **simulation** : on fabrique deux groupes de 40 000 personnes, avec la dispersion de nos salaires, et un écart réel imposé.

In [ ]:
rng = np.random.default_rng(2026)
n, sd = 40000, 1465

for ecart_reel in [3, 25]:
    a = rng.normal(2600,              sd, n)
    b = rng.normal(2600 - ecart_reel, sd, n)
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    print(f"ecart reel de {ecart_reel:>2} euros : p = {p:.3f}")

Avec 40 000 personnes par groupe, un écart de **25 €** par mois — 1 % du salaire — est statistiquement significatif. Il n'a aucune portée économique.

À l'inverse, 3 € ne sont pas détectables : il faudrait près de deux millions de personnes par groupe.

| Question | Réponse fournie par |
|---|---|
| L'écart est-il distinguable de zéro ? | la p-value |
| De quelle taille est-il ? | l'estimation ponctuelle |
| Avec quelle précision le connaît-on ? | l'intervalle de confiance |
| Est-il important ? | **l'économie, pas la statistique** |

## 6. Deux variables qualitatives

Le temps partiel dépend-il du sexe ? `TEMPS` vaut 1 pour le temps complet, 2 pour le temps partiel, et 9 pour la non-réponse — qu'on écarte.

In [ ]:
d = df[df["TEMPS"] != 9]
tableau = pd.crosstab(d["SEXE"], d["TEMPS"])

chi2 = stats.chi2_contingency(tableau)
print("khi-deux :", round(chi2.statistic, 1), "| p-value :", f"{chi2.pvalue:.1e}")
tableau

Le test rejette l'indépendance avec une p-value de l'ordre de 10⁻⁶⁷. Il dit que les deux variables sont **liées** — et rien d'autre : ni dans quel sens, ni avec quelle force, ni pourquoi.

Sur un grand échantillon, il rejette presque toujours, pour la même raison qu'un écart de 25 € devient significatif. Ce qui est plus informatif, ce sont les **proportions conditionnelles** :

In [ ]:
(100 * pd.crosstab(d["SEXE"], d["TEMPS"], normalize="index")).round(1)

**32 %** des femmes travaillent à temps partiel, contre **14 %** des hommes : plus du double. Le tableau parle davantage que le test.

Et il éclaire la partie 5 : une partie de l'écart de salaire mensuel tient simplement au nombre d'heures travaillées.

---

Supports, données et corrigés : `stefaniamarcassa.github.io/analyse_des_donnees`